In [ ]:
import os
os.environ["LD_LIBRARY_PATH"] = ""


In [ ]:
import os
import gc
import cv2
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Set CaImAn working directory
os.environ['CAIMAN_DATA'] = r"/home/toor/Desktop/aleks_temp/caiman_data"
Path(os.environ['CAIMAN_DATA']).mkdir(parents=True, exist_ok=True)

import caiman as cm
from caiman.motion_correction import MotionCorrect
from caiman.source_extraction.cnmf import cnmf, params

In [ ]:
# -------------------------
# Input AVI file
# -------------------------
video_path = Path(r"/home/toor/Desktop/toBeDeleted/processed_video.avi")
assert video_path.exists(), f"Could not find file: {video_path}"

# -------------------------
# Acquisition settings
# -------------------------
fr = 25
decay_time = 0.56
dxy = (0.83, 0.83)

# -------------------------
# Motion correction settings
# -------------------------
strides = (64, 64)
overlaps = (32, 32)
max_shifts = (25, 25)
max_deviation_rigid = 8
gSig_filt = (6, 6)
pw_rigid = True

# -------------------------
# CNMF settings
# -------------------------
p = 1
gSig = (4, 4)
merge_thr = 0.65
rf = 32
stride_cnmf = 16
ssub = 1
tsub = 1
gnb = 0
min_corr = 0.8
min_pnr = 6.5
ssub_B = 2

# -------------------------
# Quality thresholds
# -------------------------
min_SNR = 2.5
rval_thr = 0.75

In [ ]:
params_dict = {
    "fnames": [str(video_path)],
    "fr": fr,
    "dxy": dxy,
    "decay_time": decay_time,

    # motion correction
    "strides": strides,
    "overlaps": overlaps,
    "max_shifts": max_shifts,
    "max_deviation_rigid": max_deviation_rigid,
    "gSig_filt": gSig_filt,
    "pw_rigid": pw_rigid,

    "p": p,
    "nb": gnb,
    "rf": rf,
    "K": None,
    "gSig": np.array(gSig),
    "gSiz": 2 * np.array(gSig) + 1,
    "stride": stride_cnmf,
    "ssub": ssub,
    "tsub": tsub,
    "merge_thr": merge_thr,
    "method_init": "corr_pnr",
    "min_corr": min_corr,
    "min_pnr": min_pnr,
    "ssub_B": ssub_B,
    "center_psf": True,
    "only_init": True,

    # quality
    "min_SNR": min_SNR,
    "rval_thr": rval_thr,
    "use_cnn": True,
    "use_cuda": False,
    
}

opts = params.CNMFParams(params_dict=params_dict)

opts.change_params({
    "quality": {
        "min_SNR": 2.5,
        "SNR_lowest": 1.0,
        "rval_thr": 0.8,
        "rval_lowest": 0.50,
        "use_cnn": True,
        "min_cnn_thr": 0.9,
        "cnn_lowest": 0.1
    }
})

print(opts)

In [ ]:
mc = MotionCorrect(
    [str(video_path)],
    dview=None,
    **opts.motion
)

mc.motion_correct(save_movie=True)

print("Motion-corrected mmap file:")
print(mc.mmap_file)

In [ ]:
border_to_0 = 0 if mc.border_nan == "copy" else mc.border_to_0

memmap_dir = Path(os.environ['CAIMAN_DATA']) / "memmaps"
memmap_dir.mkdir(parents=True, exist_ok=True)

fname_new = cm.save_memmap(
    mc.mmap_file,
    base_name=str(memmap_dir / "memmap_"),
    order="C",
    border_to_0=border_to_0,
    dview=None
)

print("Saved CNMF memmap:", fname_new)

In [ ]:
Yr, dims, T = cm.load_memmap(fname_new)
images = np.reshape(Yr.T, [T] + list(dims), order="F")

print("Movie shape:", images.shape)   # (T, X, Y)

In [ ]:
plt.figure(figsize=(6, 6))
plt.imshow(images[0], cmap="gray")
plt.title("First corrected frame")
plt.axis("off")
plt.show()

In [ ]:
cnm = cnmf.CNMF(
    n_processes=1,
    params=opts,
    dview=None
)

cnm = cnm.fit(images)

print("CNMF fit complete")

In [ ]:
Cn, pnr = cm.summary_images.correlation_pnr(
    images[::max(T // 1000, 1)],
    gSig=opts.init["gSig"][0],
    swap_dim=False
)

cnm.estimates.Cn = Cn

cnm.estimates.evaluate_components(
    images,
    cnm.params,
    dview=None
)

print("Accepted:", len(cnm.estimates.idx_components))
print("Rejected:", len(cnm.estimates.idx_components_bad))

In [ ]:
cnm.estimates.detrend_df_f(
    quantileMin=8,
    frames_window=250,
    flag_auto=False,
    use_residuals=False,
    detrend_only=True
)

print("Detrending complete")

In [ ]:
plt.figure(figsize=(7, 7))
plt.imshow(Cn, cmap="gray")
plt.title("Correlation image")
plt.axis("off")
plt.show()

In [ ]:
output_dir = video_path.parent / "caiman_final"
output_dir.mkdir(parents=True, exist_ok=True)

results_path = output_dir / "caiman_results.hdf5"
cnm.save(str(results_path))

print("Saved results to:", results_path)

In [ ]:
if cnm.estimates.C is not None and cnm.estimates.C.shape[0] > 0:
    plt.figure(figsize=(12, 4))
    n_show = min(5, cnm.estimates.C.shape[0])
    for i in range(n_show):
        plt.plot(cnm.estimates.C[i] + i * 5)
    plt.title("Example extracted traces")
    plt.xlabel("Frame")
    plt.show()
else:
    print("No traces found")

In [ ]:
gc.collect()
print("Done")